# Question 3 - Traffic Flow Time Series

This notebook uses the fine-tuned detector from Exercise 2 to estimate traffic entering the junction from four directions. It then builds time series and compares ARIMA/SARIMAX forecasting with a naive persistence baseline.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Exercise_3":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPT = PROJECT_ROOT / "Exercise_3" / "exercise3_flow_forecasting.py"
WEIGHTS = PROJECT_ROOT / "Exercise_2" / "runs" / "train" / "yolov8n_zhandong_v2" / "weights" / "best.pt"
RESULTS_DIR = PROJECT_ROOT / "Exercise_3" / "results"
DEVICE = "0"

print("Project:", PROJECT_ROOT)
print("Script:", SCRIPT)
print("Detector weights:", WEIGHTS)
print("Weights exist:", WEIGHTS.exists())

## 1. Check Counting Lines

Run this first. It only draws the four incoming-road counting lines, so it is safe to run before the Exercise 2 model finishes. If the overlay looks wrong, edit `Exercise_3/counting_lines.json`.

In [ ]:
subprocess.run([
    sys.executable, str(SCRIPT),
    "--overlay-only",
    "--source", "zhandong_road1",
], check=True)

print("Overlay:", RESULTS_DIR / "counting_lines_overlay_zhandong_road1.jpg")

## 2. Run Detector, Tracking, Counting, and Forecasting

Run this after Exercise 2 training is complete and `best.pt` exists.

In [ ]:
assert WEIGHTS.exists(), "Run Exercise 2 training first; best.pt is missing."

subprocess.run([
    sys.executable, str(SCRIPT),
    "--source", "zhandong_road1",
    "--device", DEVICE,
    "--batch", "16",
    "--imgsz", "960",
    "--conf", "0.25",
], check=True)

## 3. Optional: Run Second Video

This produces the same outputs for `zhandong_road2`.

In [ ]:
subprocess.run([
    sys.executable, str(SCRIPT),
    "--source", "zhandong_road2",
    "--device", DEVICE,
    "--batch", "16",
    "--imgsz", "960",
    "--conf", "0.25",
], check=True)

## 4. Inspect Results

Use these files in the report:

- `Exercise_3/results/counting_lines_overlay_zhandong_road1.jpg`
- `Exercise_3/results/flow_10s_zhandong_road1.csv`
- `Exercise_3/results/forecast_metrics_zhandong_road1.csv`
- `Exercise_3/results/zhandong_road1/traffic_flow_timeseries.png`
- `Exercise_3/results/zhandong_road1/forecast_total_flow.png`
- `Exercise_3/results/zhandong_road1/forecast_west_in.png`, `forecast_east_in.png`, `forecast_north_in.png`, `forecast_south_in.png`

In [ ]:
summary_path = RESULTS_DIR / "summary_zhandong_road1.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
else:
    print("Summary not created yet. Run the detector/counting cell first.")